# Μάθημα 02 - Εξερευνώντας το Microsoft Agent Framework

Το **Microsoft Agent Framework (MAF)** είναι ένα ενιαίο πλαίσιο για τη δημιουργία πρακτόρων AI. Παρέχει μια καθαρή, συνθετή αρχιτεκτονική με τέσσερα βασικά δομικά στοιχεία:

- **Πελάτης (Client)** – συνδέεται σε ένα endpoint μοντέλου AI και διαχειρίζεται την επικοινωνία
- **Πράκτορας (Agent)** – περιβάλλει έναν πελάτη με οδηγίες και ορισμούς εργαλείων
- **Εργαλεία (Tools)** – επεκτείνουν τις δυνατότητες του πράκτορα με προσαρμοσμένες συναρτήσεις που μπορεί να καλέσει το μοντέλο
- **Συνεδρία (Session)** – διατηρεί το ιστορικό συνομιλίας για αλληλεπιδράσεις πολλαπλών γύρων

Σε αυτό το μάθημα, θα δημιουργήσουμε έναν **πράκτορα κράτησης ταξιδιών** που ελέγχει τη διαθεσιμότητα προορισμών χρησιμοποιώντας αυτές τις έννοιες.


## Ρύθμιση


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Κατανόηση της Αρχιτεκτονικής του Πλαισίου Agent

Το Πλαίσιο Agent της Microsoft ακολουθεί μια αρχιτεκτονική πολλαπλών επιπέδων:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Πελάτης** – Ένας `FoundryChatClient` συνδέεται με μια ανάπτυξη Azure OpenAI. Διαχειρίζεται την αυθεντικοποίηση, τη μορφοποίηση αιτήματος και την ανάλυση απάντησης.
2. **Agent** – Δημιουργείται από τον πελάτη μέσω του `provider.create_agent()`, ο agent συνδυάζει την πρόσβαση στο μοντέλο με οδηγίες (system prompt) και εργαλεία.
3. **Εργαλεία** – Συναρτήσεις Python διακοσμημένες με το `@tool` που ο agent μπορεί να καλέσει για να εκτελέσει ενέργειες ή να ανακτήσει δεδομένα.
4. **Σύνοδος** – Ένα αντικείμενο `AgentSession` (δημιουργημένο μέσω `agent.create_session()`) που αποθηκεύει το ιστορικό της συνομιλίας, επιτρέποντας διάλογο πολλαπλών γύρων όπου ο agent θυμάται το προηγούμενο πλαίσιο.

Ας χτίσουμε κάθε επίπεδο βήμα προς βήμα.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Προσθήκη Εργαλείων με το Διακοσμητή @tool

Τα εργαλεία επιτρέπουν στους πράκτορες να εκτελούν ενέργειες πέρα από την παραγωγή κειμένου. Ο διακοσμητής `@tool` μετατρέπει μια κανονική συνάρτηση Python σε κάτι που ο πράκτορας μπορεί να καλέσει.

Βασικά σημεία:
- Χρησιμοποιήστε `Annotated[type, "description"]` ώστε το μοντέλο να κατανοεί κάθε παράμετρο.
- Το docstring γίνεται η περιγραφή του εργαλείου που βλέπει το μοντέλο.
- Το `approval_mode="never_require"` σημαίνει ότι το εργαλείο εκτελείται αυτόματα χωρίς επιβεβαίωση από τον χρήστη.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Δημιουργία ενός Αντιπροσώπου με Εργαλεία

Τώρα συνδυάζουμε τον πελάτη, τις οδηγίες και τα εργαλεία σε έναν αντιπρόσωπο. Οι `οδηγίες` λειτουργούν ως το σύστημα προτροπής — ορίζουν την προσωπικότητα και τη συμπεριφορά του αντιπροσώπου.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Συζητήσεις με πολλούς γύρους χρησιμοποιώντας Συνεδρίες

Μια `AgentSession` (δημιουργείται μέσω `agent.create_session()`) παρακολουθεί όλα τα μηνύματα σε μια συζήτηση. Με το να περνάμε την ίδια συνεδρία σε κάθε κλήση `agent.run()`, ο πράκτορας έχει πρόσβαση σε ολόκληρο το ιστορικό της συζήτησης και μπορεί να αναφερθεί σε παλαιότερα μηνύματα.

Περνάμε `tools=[check_destination_availability]` ώστε ο πράκτορας να μπορεί να καλεί τον έλεγχο διαθεσιμότητας μας σε κάθε γύρο.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Περίληψη

Σε αυτό το μάθημα εξερευνήσατε τους τέσσερις πυλώνες του Microsoft Agent Framework:

| Έννοια | Τι Μάθατε |
|---------|------------------|
| **Πελάτης** | `FoundryChatClient` συνδέεται με το Azure OpenAI με αυθεντικοποίηση με βάση διαπιστευτήρια |
| **Πράκτορας** | `provider.create_agent()` δένει μια σύνδεση μοντέλου με οδηγίες και ένα όνομα |
| **Εργαλεία** | Ο διακοσμητής `@tool` εκθέτει συναρτήσεις Python για να καλεί ο πράκτορας |
| **Συνεδρία** | `agent.create_session()` διατηρεί το ιστορικό της συνομιλίας σε πολλούς γύρους |

Αυτά τα δομικά στοιχεία συνδυάζονται για να δημιουργήσουν πράκτορες που μπορούν να έχουν φυσικές συνομιλίες, να καλούν εξωτερικές συναρτήσεις και να διατηρούν το πλαίσιο — το θεμέλιο για πιο προηγμένα μοτίβα πρακτόρων σε επόμενα μαθήματα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
